### Figure S6A. Benchmarking UHVDB processing of UHGV HQ HC viruses

### Benchmarking UHVDB filtering of UHGV HQ HC viruses

In [1]:
### Load packages
import polars as pl

In [2]:
### Load UHGV metadata
# !wget https://portal.nersc.gov/UHGV/metadata/uhgv_metadata.tsv

uhgv_metadata = pl.read_csv("uhgv_metadata.tsv", separator='\t', null_values=['NA'], ignore_errors=True)

In [3]:
### Print UHGV stats
print(f"Total UHGV genomes: {uhgv_metadata.height}")
print(f"Total UHGV HQ genomes: {uhgv_metadata.filter(pl.col('checkv_completeness') >= 90).height}")
uhgv_hq_hc_metadata = uhgv_metadata.filter(pl.col('checkv_completeness') >= 90).filter(pl.col('viral_confidence') == 'Confident')
print(f"Total UHGV HQ HC genomes: {uhgv_hq_hc_metadata.height}")

Total UHGV genomes: 873994
Total UHGV HQ genomes: 212415
Total UHGV HQ HC genomes: 208604


In [4]:
### Get UHVDB stats for analyzing UHGV HQ HC genomes
classify_df = (
    pl.read_csv('../figure_1/uhgv_hq_hc_results/2026-03-11_outputs/classify/new_classify.tsv.gz', separator='\t')
        # .filter(pl.col('uhvdb_virus_classification').is_in(['confident', 'uncertain']))
        .with_columns([
            pl.col('seq_name').str.replace(r'\|provirus.*', '')
        ])
)

classify_viruses = (
    classify_df
        .filter(pl.col('uhvdb_virus_classification').is_in(['confident', 'uncertain']))
)
print("Number of UHGV genomes in geNomad virus summary files:", classify_df.height)
print("Number of UHGV genomes classified as confident or uncertain viruses:", classify_viruses.height)
### Filters:
# 1. length >= 2000 bp
# 2. virus score >= 0.7 OR taxonomßy contains "Inoviridae"
# 3. if taxonomy contains "Caudoviricetes", length >= 10000 bp
# 4. if taxonomy contains "Inoviridae", length >= 4500 bp and <= 12500 bp
# 5. taxonomy is not "Unclassified"
# 6. taxonomy contains "viricetes" (Class assignment) OR "Anelloviridae"
# 7. CheckV AAI completeness >= 50%
# 8. CheckV kmer_frequency <= 1.2
# 9. Contig length <= 1.5x expected CheckV length

Number of UHGV genomes in geNomad virus summary files: 208517
Number of UHGV genomes classified as confident or uncertain viruses: 207674


In [5]:
uhgv_hq_hc_not_viral = set(
    uhgv_hq_hc_metadata
        .filter(~pl.col('uhgv_genome').is_in(set(classify_viruses['seq_name'])))
        ['uhgv_genome']
)

(
    classify_df
        .filter(pl.col('seq_name').is_in(uhgv_hq_hc_not_viral))
        .group_by('taxonomy')
        .len()
        .sort('len', descending=True)
)

taxonomy,len
str,u32
"""Viruses;Monodnaviria;Sangervir…",440
"""Viruses;;;;;;Anelloviridae""",253
"""Viruses;Monodnaviria;Shotokuvi…",57
"""Viruses;Monodnaviria;Shotokuvi…",21
"""Viruses;Monodnaviria;Shotokuvi…",20
…,…
"""Viruses;Monodnaviria;Shotokuvi…",2
"""Viruses;Monodnaviria;Shotokuvi…",1
"""Unclassified""",1


In [6]:
### Identify number of viruses discarded by length
print("Number of UHGV HQ HC genomes < 2000 bp:",
    uhgv_hq_hc_metadata
        .filter(pl.col('genome_length') < 2000).height
)

Number of UHGV HQ HC genomes < 2000 bp: 70


In [9]:
### Write out IDS of HQ viruses
!zgrep "^>" ../figure_1/uhgv_hq_hc_results/2026-03-11_outputs/hqfilter/hq_viruses.fna.gz | \
    sed 's/^>//g' > uhgv_hq_hc_hqfilter_ids.txt

In [7]:
### Read in HQ virus IDs to identify number of seq passing (with manual DTRs)
hq_virus_ids = set(pl.read_csv('uhgv_hq_hc_hqfilter_ids.txt', has_header=False)['column_1'])
print("Number of UHGV HQ HC genomes passing virus and HQ filters:", len(hq_virus_ids))

Number of UHGV HQ HC genomes passing virus and HQ filters: 201794


In [8]:
### Investigate completeness method of virusees not passing HQ filters
(
    uhgv_hq_hc_metadata
        .filter(~pl.col('uhgv_genome').is_in(hq_virus_ids))
        .group_by('checkv_completeness_method')
        .len()
)

checkv_completeness_method,len
str,u32
"""AAI-based""",5130
"""Provirus""",175
"""DTR""",1235
"""HMM-based""",24
"""ITR""",345


In [11]:
### Load UHVDB hcfilter viruses
seqhash = (
    pl.read_csv('../figure_1/uhgv_hq_hc_results/2026-03-11_outputs/hcfilter/hq_hc_viruses/hq_hc_viruses.seqhasher.tsv.gz', separator='\t', columns=['original_id', 'hash'])
        .with_columns(pl.col('original_id').str.replace(r'\|provirus.*', ''))
)

print("Number of UHGV HQ HC genomes passing virus, HQ, and HC filters:", seqhash.height)
print("Proportion of UHGV genomes passing virus, HQ, and HC filters:", seqhash.height / uhgv_hq_hc_metadata.height)
print("Number of unique hashes for passing genomes:", seqhash.unique('hash').height)

Number of UHGV HQ HC genomes passing virus, HQ, and HC filters: 200906
Proportion of UHGV genomes passing virus, HQ, and HC filters: 0.9630975436712623
Number of unique hashes for passing genomes: 179017


In [12]:
### Load UHVDB genomovar clusters
genomovars = (
    pl.read_csv('../figure_1/uhgv_hq_hc_results/2026-03-11_outputs/genomovar_cluster/hq_virus_genomovars.ani_reps_info.tsv.gz', separator='\t')
        [['seq_name', 'votu_rep']]
        .with_columns([
            pl.col('seq_name').str.replace(r'\|provirus.*', ''),
            pl.col('votu_rep').str.replace(r'\|provirus.*', ''),
        ])
        .rename({'votu_rep': 'genomovar_rep'})
)
print("Number of genomovars:", genomovars.unique('genomovar_rep').height)

Number of genomovars: 156061


### Benchmarking UHVDB ANI clustering of UHGV HQ HC viruses

In [13]:
### Load UHVDB species clusters
species = (
    pl.read_csv('../figure_1/uhgv_hq_hc_results/2026-03-11_outputs/species_cluster/species.ani_reps_info.tsv.gz', separator='\t')
        .with_columns([
            pl.col('seq_name').str.replace(r'\|provirus.*', ''),
        ])
        .rename({'seq_name': 'genomovar_rep'})
)
print("Number of UHVDB species clusters:", species.unique('votu_rep').height)

Number of UHVDB species clusters: 53524


In [25]:
### Create mapping file
mapping = (
    pl.read_csv('../figure_1/uhgv_hq_hc_results/2026-03-11_outputs/rename/hq_hc_viruses/hq_hc_viruses.id_mapping.tsv.gz', new_columns=['original_id', 'seq_name'], separator='\t', ignore_errors=True, has_header=False)
    .with_columns([
        pl.col('original_id').str.replace(r'\|provirus.*', '')
    ])
)

seqhash_reps = (
    seqhash
        .join(mapping, on='original_id', how='inner')
        .filter(pl.col('seq_name').is_in(set(genomovars['seq_name'])))
        .rename({'original_id': 'seqhash_rep'})
)


joined = (
    seqhash
        .join(seqhash_reps, on='hash', how='inner')
        .join(genomovars, on='seq_name', how='inner')
        .join(species[['genomovar_rep', 'votu_rep']], on='genomovar_rep', how='inner')
)

In [26]:
### Identify UHGV genomes in UHVDB HQ HC genomovars
uhgv_in_uhvdb_filtered = (
    uhgv_hq_hc_metadata
        .filter(
            pl.col('uhgv_genome').is_in(set(joined['original_id']))
        )
)
print("Number of UHGV genomes represented in UHVDB HQ HC genomovar reps:", uhgv_in_uhvdb_filtered.unique('uhgv_genome').height)
print("Number of UHGV vOTUs represented in UHVDB HQ HC genomovar reps:", uhgv_in_uhvdb_filtered.unique('uhgv_votu').height)
print("Number of UHGV vOTU reps represented in UHVDB HQ HC genomovars:", uhgv_in_uhvdb_filtered.filter(pl.col('votu_representative') == 'Yes').height)

Number of UHGV genomes represented in UHVDB HQ HC genomovar reps: 200906
Number of UHGV vOTUs represented in UHVDB HQ HC genomovar reps: 56000
Number of UHGV vOTU reps represented in UHVDB HQ HC genomovars: 55267


In [27]:
### Map UHGV genomes to UHGV votu reps
uhgv_votu_reps = (
    uhgv_in_uhvdb_filtered
        .filter(pl.col('votu_representative') == 'Yes')
        [['uhgv_genome', 'uhgv_votu', 'genome_length', 'checkv_viral_markers', 'checkv_completeness_method']]
        .rename({'uhgv_genome': 'votu_rep'})
        .join(uhgv_in_uhvdb_filtered, on='uhgv_votu', how='right')
        [['uhgv_genome', 'uhgv_votu', 'votu_rep', 'genome_length', 'checkv_viral_markers', 'checkv_completeness_method']]
        .rename({'votu_rep': 'uhgv_votu_rep', 'genome_length': 'uhgv_votu_rep_genome_length', 'checkv_viral_markers': 'uhgv_votu_rep_viral_markers', 'checkv_completeness_method': 'uhgv_votu_rep_completeness_method'})
)

In [29]:
### Compare UHVDB selected vOTU reps with those from UHGV
uhvdb_votu_reps = (
    joined
        .join(species[['genomovar_rep', 'aai_expected_length', 'length', 'viral_genes', 'completeness_method']], on='genomovar_rep', how='inner')
        .join(uhgv_votu_reps, left_on='original_id', right_on='uhgv_genome', how='inner')
        .filter(
            (pl.col('genomovar_rep') == pl.col('votu_rep'))
        )
        .unique('votu_rep')
)
print("Total number of UHVDB vOTU reps:", uhvdb_votu_reps.height)

### Identify exact votu rep matches
uhvdb_uhgv_exact_match = uhvdb_votu_reps.filter(pl.col('original_id') == pl.col('uhgv_votu_rep')).height
print("Number of UHVDB vOTU reps that are also UHGV vOTU reps:", uhvdb_uhgv_exact_match)

### identify votu rep length/viral gene count matches
uhvdb_uhgv_len_match = (
    uhvdb_votu_reps
        .filter(
            (pl.col('votu_rep') == pl.col('uhgv_votu_rep')) |
            (pl.col('length') == pl.col('uhgv_votu_rep_genome_length')) 
        )
        .height
)
print("Number of UHVDB vOTU reps that are also UHGV vOTU reps (or same length + same virus gene count):", uhvdb_uhgv_len_match)

### Calculate total agreement
uhgv_votu_reps_in_uhvdb = uhgv_in_uhvdb_filtered.filter(pl.col('votu_representative') == 'Yes').height
print("Number of UHGV vOTU reps represented in UHVDB HQ HC viruses:", uhgv_votu_reps_in_uhvdb)
print("Agreement:", uhvdb_uhgv_len_match / uhgv_votu_reps_in_uhvdb)

Total number of UHVDB vOTU reps: 53524
Number of UHVDB vOTU reps that are also UHGV vOTU reps: 42687
Number of UHVDB vOTU reps that are also UHGV vOTU reps (or same length + same virus gene count): 48523
Number of UHGV vOTU reps represented in UHVDB HQ HC viruses: 55267
Agreement: 0.8779741979843306


In [30]:
### calculate homogeneity and completeness scores for UHGV cluster subsets
from sklearn.metrics.cluster import homogeneity_score
from sklearn.metrics.cluster import completeness_score
from sklearn.metrics import v_measure_score
import polars as pl

uhgv_v_uhvdb_results = []

clusters_full = (
    joined
        .with_columns([
            pl.col('votu_rep').str.split('-').list[1].alias('cluster_id')
        ])
        .sort(pl.col('original_id'), descending=False)
)

uhgv_clusters = (
    pl.read_csv('uhgv_metadata.tsv', separator='\t',
        columns=['uhgv_genome', 'uhgv_votu']
    )
    .filter(
        (pl.col('uhgv_genome').is_in(set(clusters_full['original_id'])))
    )
    .with_columns([
        pl.col('uhgv_votu').str.split('-').list[1].alias('uhgv_votu')
    ])
    .sort(pl.col('uhgv_genome'), descending=False)
    [['uhgv_genome', 'uhgv_votu']]
)

multi_uhgv_clusters = set(uhgv_clusters.group_by('uhgv_votu').len().filter(pl.col('len') > 1)['uhgv_votu'])
uhgv_clusters_filt = uhgv_clusters.filter(pl.col('uhgv_votu').is_in(multi_uhgv_clusters))

multi_seq_clusters = set(clusters_full.group_by('cluster_id').len().filter(pl.col('len') > 1)['cluster_id'])
clusters_full_filt = clusters_full.filter(pl.col('cluster_id').is_in(multi_seq_clusters))

# Homogeneity: measure of how often genomes in new clusters were also clustered together in prior clusters
h_score = homogeneity_score(uhgv_clusters['uhgv_votu'], clusters_full['cluster_id'])

# Completeness: measure of how often UHGV genomes that were clustered together in prior clusters are also clustered together in new clusters
c_score = completeness_score(uhgv_clusters['uhgv_votu'], clusters_full['cluster_id'])

# calculate v-measure score
v_score = v_measure_score(uhgv_clusters['uhgv_votu'], clusters_full['cluster_id'])

uhgv_v_uhvdb_results.append({
    "num_uhgv_clusters": uhgv_clusters['uhgv_votu'].n_unique(),
    "num_multi_uhgv_clusters": uhgv_clusters_filt['uhgv_votu'].n_unique(),
    "num_in_multi_uhgv_clusters": uhgv_clusters_filt.shape[0],
    "num_uhvdb_clusters": clusters_full['cluster_id'].n_unique(),
    "num_multi_uhvdb_clusters": clusters_full_filt['cluster_id'].n_unique(),
    "num_in_multi_uhvdb_clusters": clusters_full_filt.shape[0],
    "h_score": h_score,
    "c_score": c_score,
    "v_score": v_score
})

In [32]:
### display results
pl.from_dicts(uhgv_v_uhvdb_results)

num_uhgv_clusters,num_multi_uhgv_clusters,num_in_multi_uhgv_clusters,num_uhvdb_clusters,num_multi_uhvdb_clusters,num_in_multi_uhvdb_clusters,h_score,c_score,v_score
i64,i64,i64,i64,i64,i64,f64,f64,f64
56000,23976,168882,53524,23106,170488,0.987754,0.996253,0.991985


In [33]:
### evaluate rep selection script using UHGV clusters
# !wget https://portal.nersc.gov/cfs/m342/UHGV/mysql/tsv/checkv_completeness.tsv
# !wget https://portal.nersc.gov/cfs/m342/UHGV/metadata/uhgv_metadata.tsv

# load cluster assignments
clusters_df = (
    pl.read_csv('uhgv_metadata.tsv', separator='\t', ignore_errors=True)
    .filter((pl.col('checkv_completeness') >= 90) & (pl.col('viral_confidence') == 'Confident'))
)
clusters_dict = dict(zip(clusters_df["uhgv_genome"], clusters_df["uhgv_votu"]))

def load_metadata(uhgv_metadata, completeness, clusters):
    df = pl.read_csv(uhgv_metadata, separator='\t', ignore_errors=True).filter((pl.col('checkv_completeness') >= 90) & (pl.col('viral_confidence') == 'Confident'))
    
    mine_report = (
        # load mine report and join with uhvdb metadata
        df
            .join(
                pl.read_csv(completeness, separator='\t', columns=['contig_id', 'aai_expected_length'], ignore_errors=True),
                how='inner', left_on='uhgv_genome', right_on='contig_id'
            )
            # retain only sequences that are in clusters
            .filter(
                (pl.col('uhgv_genome').is_in(clusters.keys()))
            )
            # create cluster_id and length columns
            .with_columns([
                pl.when(pl.col('genome_length').is_not_null())
                    .then(pl.col('genome_length'))
                    .alias('length').cast(pl.Float64)
            ])
            .with_columns([pl.col('uhgv_genome').replace_strict(clusters, default=None).alias('cluster_id')])
    )

    return mine_report


# load sequence metadata
mine_report = load_metadata('uhgv_metadata.tsv', 'checkv_completeness.tsv', clusters_dict)

# 1. calculate median length amd size each cluster
cluster_metrics = (
    mine_report.group_by('cluster_id').agg(
        [
            pl.col('length').median().alias('median_length'),
            pl.col('checkv_viral_markers').max().alias('max_viral_genes'),
            pl.col('uhgv_genome').len().alias('num_seqs')
        ]
    )
)

mine_report_metrics = (
    mine_report
        .join(cluster_metrics, on='cluster_id', how='inner')
)

# 2. assign singletons as vOTU representatives
singleton_clusters = set(
    cluster_metrics.filter(pl.col('num_seqs') == 1)['cluster_id']
)

cluster_reps = (
    mine_report
        .filter(pl.col('cluster_id').is_in(singleton_clusters))['uhgv_genome', 'cluster_id']
)

# 3. assign longest DTRs as vOTU representatives (if > median length)
dtr_cluster_reps = (
    mine_report_metrics
        .filter(
            (
                (pl.col('checkv_completeness_method').str.contains('DTR'))
            ) &
            ~(pl.col('cluster_id').is_in(set(cluster_reps['cluster_id'])))
        )
        .sort(['length', 'checkv_viral_markers'], descending=True)
        .group_by('cluster_id', maintain_order=True)
        .first()['uhgv_genome', 'cluster_id']
)

cluster_reps = pl.concat([cluster_reps, dtr_cluster_reps])

# 4. Assign linear genome closest to expected AAI length with highest number of viral genes
linear_cluster_reps = (
    mine_report_metrics
        .filter(
            ~(pl.col('cluster_id').is_in(set(cluster_reps['cluster_id']))) &
            (pl.col('checkv_viral_markers') == pl.col('max_viral_genes'))
        )
        .with_columns([
            pl.col('aai_expected_length').cast(pl.String).str.replace('NA', pl.col('median_length')).cast(pl.Float64).alias('aai_expected_length'),
        ])
        .with_columns([
            (abs(pl.col('genome_length').cast(pl.Float64) - pl.col('aai_expected_length').cast(pl.Float64))).alias('length_diff'),
        ])
        .sort(pl.col('length_diff'), descending=False)
        .group_by('cluster_id', maintain_order=True)
        .first()['uhgv_genome', 'cluster_id']
)

cluster_reps = pl.concat([cluster_reps, linear_cluster_reps])

In [34]:
cluster_metrics = (
    mine_report_metrics
        [['uhgv_genome', 'cluster_id', 'num_seqs', 'length', 'median_length', 'aai_expected_length', 'checkv_viral_markers', 'max_viral_genes', 'checkv_completeness_method']]
        .join(cluster_reps, on='cluster_id', how='full', suffix='_rep')
        .drop('cluster_id_rep')
        .rename({'uhgv_genome_rep': 'votu_rep'})
)

cluster_metrics.sort(pl.col('cluster_id'), descending=False)
print("Total number of HQ HC vOTUs in UHGV:", mine_report['cluster_id'].n_unique())

uhgv_metadata = pl.read_csv('uhgv_metadata.tsv', separator='\t', ignore_errors=True)

new_v_old = (
    cluster_metrics
        .filter(pl.col('uhgv_genome') == pl.col('votu_rep'))
        .join(uhgv_metadata.filter(pl.col('votu_representative') == 'Yes'), left_on='cluster_id', right_on='uhgv_votu', how='left', suffix='_old')
)

exact = new_v_old.filter(pl.col('votu_rep') == pl.col('uhgv_genome_old')).height
print("Number of exact matches:", exact)

same_length = (   
   new_v_old
        .filter(
            (pl.col('votu_rep') == pl.col('uhgv_genome_old')) |
            (
                (pl.col('length') == pl.col('genome_length')) &
                (pl.col('checkv_viral_markers') == pl.col('checkv_viral_markers_old'))
            )
        )
        .shape[0]
)
print("Number of non-matches with same length & viral gene count:", same_length)

print("Prop agreement:", same_length / new_v_old.height)

Total number of HQ HC vOTUs in UHGV: 58044
Number of exact matches: 50370
Number of non-matches with same length & viral gene count: 53624
Prop agreement: 0.9238508717524636


### Benchmarking UHVDB AAI clustering of UHGV sequences

In [35]:
import gzip
### function to load mcl clusters into a DataFrame
def load_mcl_clusters(mcl, seq_id):
    # assign sequences to mcl clusters
    clusters = {}

    cluster_id = 0
    with gzip.open(mcl, 'rt') as mcl_file:
        for line in mcl_file:
            cluster_id += 1
            for node in line.strip().split():
                clusters[node] = cluster_id

    # assign unclustered sequences to their own cluster
    with open(seq_id, 'r') as seqid_file:
        for line in seqid_file:
            sequence = line.strip()
            if sequence not in clusters:
                cluster_id += 1
                clusters[sequence] = cluster_id

    # convert to a DataFrame
    clusters_df = pl.DataFrame({
        'seq_id': list(clusters.keys()),
        'cluster_id': list(clusters.values())
    })

    return clusters_df

In [38]:
### calculate homogeneity and completeness scores for UHVDB clusters vs UHGV
from sklearn.metrics.cluster import homogeneity_score
from sklearn.metrics.cluster import completeness_score
from sklearn.metrics import v_measure_score
import polars as pl

# !wget https://portal.nersc.gov/cfs/m342/UHGV/metadata/votus_metadata_extended.tsv

# create IDs file from ANI reps
(
    species
        .unique('votu_rep')
        [['votu_rep']]
        .write_csv('ani_reps_ids.tsv', include_header=False)
)

uhgv_v_uhvdb_results = []

for rank in ['family', 'subfamily', 'genus', 'subgenus']:
    print(f"Calculating scores for {rank}...\n")

    if rank == 'family':
        clusters_full = (
            load_mcl_clusters('../figure_1/tmp/2026-03-11/aaicluster/mcl/family_normscore/family_normscore.mcl.gz','ani_reps_ids.tsv')
                .join(joined, left_on='seq_id', right_on='seq_name', how='inner')
                .sort(pl.col('original_id'), descending=False)
        )
    else:
        clusters_full = (
            load_mcl_clusters(f'../figure_1/tmp/2026-03-11/aaicluster/mclcluster/{rank}/mcl/{rank}/{rank}.mcl.gz','ani_reps_ids.tsv')
                .join(joined, left_on='seq_id', right_on='seq_name', how='inner')
                .sort(pl.col('original_id'), descending=False)
        )

    uhgv_clusters = (
        pl.read_csv('votus_metadata_extended.tsv', separator='\t',
            columns=['uhgv_genome', 'uhgv_taxonomy', 'checkv_completeness', 'viral_confidence'], null_values=['NA', 'NULL']
        )
        .filter(pl.col('uhgv_taxonomy').str.contains(r'vFAM.*vSUBFAM.*vGENUS.*vSUBGEN.*vOTU.*'))
        .with_columns([
            pl.col('uhgv_taxonomy').str.split(';').list[0].str.split('-').list[1].alias('uhgv_family'),
            pl.col('uhgv_taxonomy').str.split(';').list[1].str.split('-').list[1].alias('uhgv_subfamily'),
            pl.col('uhgv_taxonomy').str.split(';').list[2].str.split('-').list[1].alias('uhgv_genus'),
            pl.col('uhgv_taxonomy').str.split(';').list[3].str.split('-').list[1].alias('uhgv_subgenus'),
            pl.col('uhgv_taxonomy').str.split(';').list[4].str.split('-').list[1].alias('uhgv_votu')
        ])
        .join(uhgv_metadata[['uhgv_genome', 'uhgv_votu']].with_columns([pl.col('uhgv_votu').str.split('vOTU-').list[-1]]), on='uhgv_votu', how='inner')
        .filter(
            (pl.col('uhgv_genome_right').is_in(set(clusters_full['original_id'])))
        )
        .sort(pl.col('uhgv_genome_right'), descending=False)
        [['uhgv_genome_right', f'uhgv_{rank}']]
    )

    clusters_full = clusters_full.filter(pl.col('original_id').is_in(set(uhgv_clusters['uhgv_genome_right']))).sort(pl.col('original_id'), descending=False)

    multi_uhgv_clusters = set(uhgv_clusters.group_by(f'uhgv_{rank}').len().filter(pl.col('len') > 1)[f'uhgv_{rank}'])
    uhgv_clusters_filt = uhgv_clusters.filter(pl.col(f'uhgv_{rank}').is_in(multi_uhgv_clusters))

    multi_seq_clusters = set(clusters_full.group_by('cluster_id').len().filter(pl.col('len') > 1)['cluster_id'])
    clusters_full_filt = clusters_full.filter(pl.col('cluster_id').is_in(multi_seq_clusters))

    # Homogeneity: measure of how often genomes in new clusters were also clustered together in prior clusters
    h_score = homogeneity_score(uhgv_clusters[f'uhgv_{rank}'], clusters_full['cluster_id'])

    # Completeness: measure of how often UHGV genomes that were clustered together in prior clusters are also clustered together in new clusters
    c_score = completeness_score(uhgv_clusters[f'uhgv_{rank}'], clusters_full['cluster_id'])

    # calculate v-measure score
    v_score = v_measure_score(uhgv_clusters[f'uhgv_{rank}'], clusters_full['cluster_id'])

    uhgv_v_uhvdb_results.append({
        "Rank": rank,
        "num_uhgv_clusters": uhgv_clusters[f'uhgv_{rank}'].n_unique(),
        "num_multi_seq_uhgv_clusters": uhgv_clusters_filt[f'uhgv_{rank}'].n_unique(),
        "num_seqs_in_multi_seq_uhgv_clusters": uhgv_clusters_filt.shape[0],
        "num_uhvdb_clusters": clusters_full['cluster_id'].n_unique(),
        "num_multi_seq_uhvdb_clusters": clusters_full_filt['cluster_id'].n_unique(),
        "num_seqs_in_multi_seq_uhvdb_clusters": clusters_full_filt.shape[0],
        "h_score": h_score,
        "c_score": c_score,
        "v_score": v_score
    })

Calculating scores for family...

Calculating scores for subfamily...

Calculating scores for genus...

Calculating scores for subgenus...



In [39]:
pl.from_dicts(uhgv_v_uhvdb_results)

Rank,num_uhgv_clusters,num_multi_seq_uhgv_clusters,num_seqs_in_multi_seq_uhgv_clusters,num_uhvdb_clusters,num_multi_seq_uhvdb_clusters,num_seqs_in_multi_seq_uhvdb_clusters,h_score,c_score,v_score
str,i64,i64,i64,i64,i64,i64,f64,f64,f64
"""family""",446,360,60914,814,749,60935,0.971747,0.785615,0.868824
"""subfamily""",6482,3676,58194,7000,4373,58373,0.993148,0.954107,0.973237
"""genus""",18907,6669,48762,18433,7591,50158,0.991337,0.977496,0.984368
"""subgenus""",38523,7071,29548,37786,7687,30901,0.991995,0.989583,0.990788
